# Compare the fixed and selected parameter configurations.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
PROJECT_DIR = PROJECT_ROOT
PYTHON_EXE = str(RELEASE_CLASSICAL_PYTHON)


In [ ]:
import subprocess,sys,os,time,json,platform
from pathlib import Path
from datetime import datetime,timezone


def python_candidates(explicit=None):
    if explicit:
        return [str(Path(explicit).expanduser().absolute())]
    candidates=[Path(sys.executable)]
    bases={Path(sys.prefix),Path(sys.base_prefix),Path(sys.executable).parent.parent,
           Path('/opt/anaconda3'),Path('/opt/miniconda3'),Path.home()/'miniconda3',Path.home()/'anaconda3'}
    for base in sorted(bases):
        candidates.extend(sorted((base/'envs').glob('*/bin/python')))
    candidates.extend(sorted((Path.home()/'.conda/envs').glob('*/bin/python')))
    registry=Path.home()/'.conda/environments.txt'
    if registry.is_file():
        candidates.extend(Path(line.strip())/'bin/python' for line in registry.read_text().splitlines() if line.strip())
    for base in [Path.cwd(),Path.cwd().parent]:
        candidates.extend([base/'.venv/bin/python',base/'venv/bin/python'])

    unique=[]
    for path in candidates:
        value=str(path.expanduser().absolute())
        if path.is_file() and value not in unique:unique.append(value)
    return unique


def compatible_python(expected,out,explicit=None):
    code="""import json,platform,importlib.metadata as m
v={'python':platform.python_version()}
for key,package in [('numpy','numpy'),('pandas','pandas'),('sklearn','scikit-learn'),('tensorflow','tensorflow')]:
 try:v[key]=m.version(package)
 except m.PackageNotFoundError:v[key]='NOT_INSTALLED'
print(json.dumps(v))
"""
    reports=[]
    for executable in python_candidates(explicit):
        item={'executable':executable}
        try:
            proc=subprocess.run([executable,'-c',code],capture_output=True,text=True,timeout=20)
            item['metadata_returncode']=proc.returncode
            if proc.returncode!=0:
                item['status']='METADATA_FAILED';item['stderr_tail']=proc.stderr[-1500:];reports.append(item);continue
            versions=json.loads(proc.stdout.strip().splitlines()[-1]);item['versions']=versions
            if any(versions.get(k)!=v for k,v in expected.items()):
                item['status']='VERSIONS_DIFFER';reports.append(item);continue
            print('Eğitim sürümleriyle eşleşen Python kontrol ediliyor:',executable,flush=True)
            probe="import numpy, pandas; import tensorflow as tf; import sklearn; print('TF_PROBE_OK',float(tf.reduce_sum(tf.constant([1.,2.])).numpy()),flush=True)"
            proc=subprocess.run([executable,'-X','faulthandler','-c',probe],capture_output=True,text=True,timeout=60)
            item.update(probe_returncode=proc.returncode,probe_stdout_tail=proc.stdout[-1500:],probe_stderr_tail=proc.stderr[-3000:])
            if proc.returncode==0 and 'TF_PROBE_OK 3.0' in proc.stdout:
                item['status']='READY';reports.append(item);return executable,reports
            item['status']='IMPORT_OR_TENSOR_TEST_FAILED'
        except subprocess.TimeoutExpired:
            item['status']='PROBE_TIMEOUT'
        except (OSError,ValueError) as e:
            item.update(status='PROBE_ERROR',error=str(e))
        reports.append(item)
    return None,reports




import hashlib,statistics,fcntl
from collections import defaultdict

def read(p):
    return json.loads(p.read_text())

def digest(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for block in iter(lambda:f.read(1048576),b''):h.update(block)
    return h.hexdigest()

def canonical(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False,separators=(',',':'),allow_nan=False).encode()).hexdigest()

def write(p,obj):
    temp=p.with_suffix('.tmp')
    temp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False))
    temp.replace(p)

def require(ok,message):
    if not ok:raise ValueError(message)

def validate(job,result,plan_hash):
    require(result['status']=='COMPLETE',result.get('error','Fold failed'))
    require(result['plan_sha256']==plan_hash and (result['seed'],result['fold'])==(job['seed'],job['fold']),'Fold identity mismatch')
    expected={(r['representation'],r['method']):r for r in job['selected_primary']}
    rows=result['rows']
    require(len(rows)==len(expected)==10 and {(r['representation'],r['method']) for r in rows}==set(expected),'Eksik/yinelenen yöntem')
    for r in rows:
        require(r['selected']==expected[r['representation'],r['method']] and (r['seed'],r['fold'])==(job['seed'],job['fold']),'Selected arm changed')
        require(r['status'] in ['VALID','UNDEFINED_REFERENCE'],'Reference status invalid')
        if r['status']=='VALID':
            require(all(0<=r['reference'][k]<=1 for k in METRICS),'Invalid reference metric')
        else:require(r['reference'] is None and r.get('reason'),'Missing undefined-case reason')

METRICS=['AP','ROC_AUC','precision','recall','F1','Macro_F1']

def aggregate(rows):
    grouped=defaultdict(list)
    for r in rows:grouped[r['representation'],r['method']].append(r)
    require(len(rows)==300 and len(grouped)==10,'300 eşleşme gerekli')
    stats=[]
    for (rep,method),group in sorted(grouped.items()):
        require(len(group)==30 and {(r['seed'],r['fold']) for r in group}=={(s,f) for s in [42,43,44] for f in range(1,11)},'Eksik fold/seed')
        valid=[r for r in group if r['status']=='VALID']
        item=dict(representation=rep,method=method,expected_pairs=30,valid_pairs=len(valid),reference_params=group[0]['reference_params'],
                  undefined=[{'seed':r['seed'],'fold':r['fold'],'reason':r['reason']} for r in group if r['status']!='VALID'])
        for m in METRICS:
            selected=[r['selected'][m] for r in group]
            item[m+'_selected_mean']=statistics.mean(selected)
            item[m+'_selected_sd']=statistics.stdev(selected)
            if len(valid)==30:
                fixed=[r['reference'][m] for r in group]
                differences=[a-b for a,b in zip(selected,fixed)]
                item[m+'_reference_mean']=statistics.mean(fixed);item[m+'_reference_sd']=statistics.stdev(fixed)
                item[m+'_paired_delta_mean']=statistics.mean(differences);item[m+'_paired_delta_sd']=statistics.stdev(differences)
                item[m+'_positive_pairs']=sum(v>1e-12 for v in differences)
                item[m+'_negative_pairs']=sum(v< -1e-12 for v in differences)
                item[m+'_tied_pairs']=sum(abs(v)<=1e-12 for v in differences)
        stats.append(item)
    return stats

def run_comparison():
    root=PROJECT_DIR/'revision_outputs'
    plan_hash=canonical(PLAN);out=root/'12_FD001_reference_vs_selected'/plan_hash[:20]
    out.mkdir(parents=True,exist_ok=True)
    summary=out/'hyperparameter_comparison_summary.json'
    report=dict(stage=PLAN['stage'],status='RUNNING',plan_sha256=plan_hash,completed_folds=0,reference_definition=PLAN['reference_definition'],AE_training_performed=False,test_files_read=False)
    with (out/'run.lock').open('a') as lock:
        fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
        try:
            require(hashlib.sha256(WORKER.encode()).hexdigest()==PLAN['worker_sha256'],'Worker kimliği farklı')
            require(canonical(read(root/PLAN['source_path']))==PLAN['source_hash'],'Önceki onaylı 05B özeti değişmiş')
            write(out/'plan.json',PLAN);worker=out/'worker.py';worker.write_text(WORKER)
            exe,reports=compatible_python(PLAN['expected_versions'],out,PYTHON_EXE)
            report.update(worker_executable=exe,environment_checks=reports)
            require(exe is not None,'Uyumlu Python bulunamadı')
            all_rows=[]
            for job in PLAN['jobs']:
                seed,fold=job['seed'],job['fold']
                folder=out/f'seed_{seed}'/f'fold_{fold:02d}';folder.mkdir(parents=True,exist_ok=True)
                marker=folder/'done.json'
                if marker.is_file():
                    pointer=read(marker);attempt=folder/pointer['attempt'];require(attempt.parent==folder,'Cache yolu geçersiz')
                    result_path=attempt/'result.json'
                    require(digest(result_path)==pointer['sha256'],'Tamamlanmış sonuç dosyası değişmiş')
                    result=read(result_path);validate(job,result,plan_hash)
                    print('KAYITTAN:',seed,fold,flush=True)
                else:
                    attempt=folder/('attempt_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'));attempt.mkdir()
                    cfg=dict(job,revision_root=str(root),data_directory=str(PROJECT_DIR/'data'),plan_sha256=plan_hash,attempt_dir=str(attempt))
                    config=attempt/'config.json';write(config,cfg);print('ÇALIŞIYOR:',seed,fold,flush=True)
                    with (attempt/'worker.log').open('w') as log:
                        proc=subprocess.Popen([exe,'-u','-X','faulthandler',str(worker),str(config)],stdout=log,stderr=subprocess.STDOUT)
                        try:code=proc.wait()
                        except BaseException:
                            proc.terminate()
                            try:proc.wait(timeout=10)
                            except subprocess.TimeoutExpired:proc.kill();proc.wait()
                            raise
                    require(code==0 and (attempt/'result.json').exists(),'Alt işlem durdu: '+str(attempt/'worker.log'))
                    result=read(attempt/'result.json')
                    if result['status']!='COMPLETE':report['failed_fold']=result
                    validate(job,result,plan_hash)
                    write(marker,dict(attempt=attempt.name,sha256=digest(attempt/'result.json')))
                all_rows.extend(result['rows']);report['completed_folds']+=1;write(summary,report)
            report['rows']=all_rows;report['aggregates']=aggregate(all_rows)
            report['status']='PASS_REFERENCE_COMPARISON' if all(r['status']=='VALID' for r in all_rows) else 'COMPLETE_WITH_UNDEFINED_REFERENCES'
            report['aggregation']='Equal-weight fold/seed means and descriptive SD; differences are selected minus fixed within the same fold and seed. Full-reference mean and contrast require 30 valid pairs.'
        except KeyboardInterrupt:report.update(status='INTERRUPTED')
        except Exception as e:
            import traceback
            report.update(status='REVIEW_REQUIRED',error=str(e),traceback=traceback.format_exc())
        finally:write(summary,report)
    print('DURUM:',report['status']);print('GÖNDERİLECEK DOSYA:',summary)
    if 'error' in report:print('HATA:',report['error'])
    return report


In [ ]:
PLAN = load_json('assets/12/03_plan.json')
WORKER = load_text('assets/12/03_worker.txt')


In [ ]:
result=run_comparison()
